# Begin

VQ-VAE to craft action plans

In [34]:
# @launchit.collected
# @launchit.collected_temp_config
# @launchit.collected_optuna
# @launchit.collected_initrd
# @launchit.collected_build_docker_launch

In [35]:
# @launchit.collected_manual_run_docker_launch
# @launchit.collected_optuna_run_docker_launch

In [36]:
import os # @launchit.collect
import sys # @launchit.collect
import socket
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import math
import datetime
import json # @launchit.collect
import pprint # @launchit.collect
import re 
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import pickle # @launchit.collect
import IPython 
from enum import StrEnum, auto # @launchit.collect
import gzip
import functools

from tqdm.notebook import tqdm

import numpy as np # @launchit.collect
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
import torch.multiprocessing as torch_mp
from torch.utils.data import TensorDataset, DataLoader

import optuna # @launchit.collect
from optuna.storages import JournalStorage # @launchit.collect
from optuna.storages.journal import JournalFileBackend # @launchit.collect
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
build_project_root_path = '${BUILD_PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
sys.path.append(os.path.join(build_project_root_path, 'lib')) # @launchit.collect
import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter, RecursiveMovingAverageFilter
from logging_utils import *
from artifact_registry import * # @launchit.collect
import torch_utils
from torch_utils import *
import launchit 
from hp_utils import * # @launchit.collect
from metrics_collector import RmqSummaryWriter, S3SummaryWriter
from autoincrement import Autoincrement
import command_listener

# Init

In [37]:
# @launchit.collect
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    DOCKER_LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

In [38]:
# @launchit.collect_temp_config
# @launchit.disable

# Construct temporary CONFIG object for build and bootstrap purposes
if '${LAUNCHIT_FNAME}' != '$' + '{LAUNCHIT_FNAME}':
    notebook_fname = '${LAUNCHIT_FNAME}'
    notebook_basename = os.path.basename(notebook_fname)
    notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    subproject_name = os.path.basename(os.path.dirname(notebook_fname))
    
    if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
        # This (bootstrap) config is for the very initial phase of docker launch when one needs to load hyperparameters.
        # This config will be recreated soon to full-fledged Config
        CONFIG = namedtuple('BootstrapConfig', 'initrd_path, exec_mode')(
            initrd_path=os.path.join(project_root_path, 'run', subproject_name, 'initrd-' + notebook_name),
            exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK,
        )
        assert Logging._instance is None, 'Must create Logging instance with use_raw_stdout=True, but Logging instance is already created'
        # use_raw_stdout is necessary because sys.stdout is hijacked by papermill and this will lead to problems
        # with string duplications written to stdout by child process (worker)
        # see https://share.google/aimode/ffIcoqyaJdgMsEcoW
        Logging.get(use_raw_stdout=True, log_fname=os.path.join(project_root_path, 'run', subproject_name, notebook_name + '.log'))
    else:
        # Config to build docker launch
        CONFIG = namedtuple('BuildConfig', 
                            'initrd_path, relative_initrd_path, relative_run_path, relative_metrics_suite_fname, ' + 
                            'model_group_uri, self_fname, relative_self_fname, self_name, subproject_name, ' + 
                            'docker_registry, exec_mode, ' +
                            'artifact_registry_cache_fname, artifact_registry_cache, artifact_registry')(
            initrd_path=os.path.join(build_project_root_path, 'run', subproject_name, 'initrd-' + notebook_name),
            relative_initrd_path=os.path.join('run', subproject_name, 'initrd-' + notebook_name), # relative to build project root
            relative_run_path=os.path.join('run', subproject_name),
            relative_metrics_suite_fname=os.path.join('run', subproject_name, notebook_name + '.metrics_suite.json'),
            model_group_uri='${MODEL_GROUP_URI}',
            self_fname=notebook_fname,
            relative_self_fname=lu.when('/run/' in notebook_fname, 'run/', '') + os.path.join(subproject_name, notebook_basename),  # relative to build project root
            self_name=notebook_name,
            subproject_name=subproject_name,
            docker_registry='cr.selcloud.ru/neurolab',
            exec_mode=ExecMode.LAUNCH_NOTEBOOK,
            artifact_registry_cache_fname=None, 
            artifact_registry_cache={}, 
            artifact_registry=None,
        )
        CONFIG = CONFIG._replace(artifact_registry=ArtifactRegistry(CONFIG.model_group_uri, cache=CONFIG.artifact_registry_cache))
        CONFIG = CONFIG._replace(artifact_registry_cache_fname=os.path.join(CONFIG.initrd_path, 'artifact_registry_cache.pkl'))
        os.makedirs(CONFIG.initrd_path, exist_ok=True)

    Logging.get()(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n')
# @launchit.stop

In [39]:
def create_config():
    config = namedtuple('Config', 
                        'host_name, ' +
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, initrd_path, ' + 
                        'self_fname, self_name, metrics_suite_fname, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, docker_registry, exec_mode, is_interactive')(
        host_name=socket.gethostname(),
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        initrd_path=None,
        self_fname=None,
        self_name=None,
        metrics_suite_fname=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cpu',
        docker_registry='cr.selcloud.ru/neurolab',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )

    env_cuda_device = os.environ.get('CUDA_DEVICE')
    
    if torch.cuda.is_available():
        default_cuda_device = 'cuda'
        config = config._replace(cuda_device=lu.coalesce(env_cuda_device, default_cuda_device))
    else:
        assert env_cuda_device is None, f'CUDA device "{env_cuda_device}" is requested but CUDA is NOT available!'
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf).get('jupyter_session')

            if notebook_fname is None:
                notebook_fname = os.path.join(config.subproject_path, os.path.basename('${LAUNCHIT_FNAME}'))
                assert os.path.exists(notebook_fname)
            
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None

            if is_launch:
                if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
                    config = config._replace(exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK)
                else:
                    config = config._replace(exec_mode=ExecMode.LAUNCH_NOTEBOOK)
            else:
                assert config.exec_mode == ExecMode.MASTER_NOTEBOOK
    
    config = config._replace(is_interactive=config.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(initrd_path=os.path.join(project_root_path, 'run', config.subproject_name, 'initrd-' + config.self_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    config = config._replace(metrics_suite_fname=os.path.join(config.run_path, config.self_name + '.metrics_suite.json'))
    return config

In [40]:
# @launchit.disable_worker
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', CONFIG.exec_mode == ExecMode.LAUNCH_MODULE)
LOG.enable('stdout', CONFIG.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
LOG.enable('verbose_stdout', CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)
LOG.enable('file', CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
LOG(f'{os.environ=}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)
os.makedirs(CONFIG.initrd_path, exist_ok=True)

CONFIG=
{'host_name': 'thinkbook',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'initrd_path': '/home/misha/dev/mine/neurolab/run/18_rl/initrd-18e_action_plan_01',
 'self_fname': '/home/misha/dev/mine/neurolab/18_rl/18e_action_plan_01.ipynb',
 'self_name': '18e_action_plan_01',
 'metrics_suite_fname': '/home/misha/dev/mine/neurolab/run/18_rl/18e_action_plan_01.metrics_suite.json',
 'subproject_name': '18_rl',
 'is_cuda': False,
 'cuda_device': 'cpu',
 'docker_registry': 'cr.selcloud.ru/neurolab',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [41]:
# @launchit.disable
# @launchit.collect
LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    launch_goal: str = None # default
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    @dataclass(slots=True)
    class General:
        comment: str = None
        random_seed: int = None
        is_torch_deterministic: bool = True

    @dataclass(slots=True)
    class Dataset:
        train: list = None # list of IDs of dataset used for training
        test: str = None # ID of a dataset used for test

    @dataclass(slots=True)
    class Model:
        parent: str = None # or dict with model names
        actions_count: int = 6
        action_plan_length: int = 10
        d_action_plan: int = 3
        fsq: dict = None # encoder specific params: levels
        encoder: dict = None # encoder specific params: features_counts
        decoder: dict = None # decoder specific params: d_inner
        is_trainable: bool = True # or dict with model names
        is_torch_compile: bool = True # or dict with model names

    @dataclass(slots=True)
    class Train:
        epochs_count: int = 1 
        batch_size: int = 32
        optimizer: str = 'AdamW'
        learn_rate: str = 'const(0.00025)'

    @dataclass(slots=True)
    class Report:
        follow_up_freq: int = 10

    general: General = dataclasses.field(default_factory=General)
    dataset: Dataset = dataclasses.field(default_factory=Dataset)
    model: Model = dataclasses.field(default_factory=Model)
    train: Train = dataclasses.field(default_factory=Train)
    report: Report = dataclasses.field(default_factory=Report)

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        hp.general = Hyperparameters.General(**hp.general)
        hp.dataset = Hyperparameters.Dataset(**hp.dataset)
        hp.model = Hyperparameters.Model(**hp.model)
        hp.train = Hyperparameters.Train(**hp.train)
        hp.report = Hyperparameters.Report(**hp.report)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()

# Runtime

## Runtime

In [42]:
@dataclass(slots=True)
class Runtime:
    command_listener: object = None
    should_save_model: bool = True
    mp_ctx: object = None
    optuna_trial: dict = None
    artifact_registry_cache: dict = None
    artifact_registry: object = None
    summary_writer: object = None
    train_dataset: object = None
    test_dataset: object = None
    model_names: tuple = dataclasses.field(default_factory=lambda: ('fsq', 'encoder', 'decoder'))
    fsq: object = None
    encoder: object = None
    decoder: object = None

## Configure

In [43]:
# @launchit.disable
# @launchit.collect
import random
HP.launch_goal = None
HP.general.comment = None
HP.general.random_seed = random.randint(0, 100)
HP.general.is_torch_deterministic = True
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': None,
 'launch_id': 0,
 'general': {'comment': None, 'random_seed': 1, 'is_torch_deterministic': True},
 'dataset': {'train': None, 'test': None},
 'model': {'parent': None,
           'actions_count': 6,
           'action_plan_length': 10,
           'd_action_plan': 3,
           'fsq': None,
           'encoder': None,
           'decoder': None,
           'is_trainable': True,
           'is_torch_compile': True},
 'train': {'epochs_count': 1,
           'batch_size': 32,
           'optimizer': 'AdamW',
           'learn_rate': 'const(0.00025)'}}


## Create

In [44]:
assert type(CONFIG).__name__ == 'Config', 'Runtime should be created for full-fledged Config instance only'
LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)

RT = Runtime()
RT.command_listener = command_listener.CommandListener()

if 'COMMAND_LISTENER' in os.environ:
    RT.command_listener.start()
    LOG(f'CommandListener started on port={RT.command_listener.DEFAULT_PORT}')
    
RT.mp_ctx = torch_mp.get_context('spawn') # Spawn is needed for CUDA, fork doesn't work within PyTorch

if HP.general.random_seed is not None:
    random.seed(HP.general.random_seed)
    torch.manual_seed(HP.general.random_seed)
    RNG = np.random.default_rng(HP.general.random_seed)    
    LOG(f'Random seed={HP.general.random_seed}')

if HP.general.is_torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.general.is_torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()

artifact_registry_type = os.environ.get('ARTIFACT_REGISTRY', '').upper()

if not artifact_registry_type:
    artifact_registry_type = lu.when(CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK, 'S3', 'NEXUS')

match artifact_registry_type:
    case 'NEXUS':
        kwargs = {}

        if os.environ.get('NEXUS_URL', None) is not None:
            kwargs['nexus_url'] = os.environ['NEXUS_URL']

        if os.environ.get('DOWNLOAD_NEXUS_URL', None) is not None:
            kwargs['download_nexus_url'] = os.environ['DOWNLOAD_NEXUS_URL']
        
        RT.artifact_registry = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri, **kwargs)
        LOG(f'ArtifactRegistry created')
    case 'S3':
        kwargs = {}
        artifact_registry_cache_fname = os.path.join(CONFIG.initrd_path, 'artifact_registry_cache.pkl')
        
        if os.path.exists(artifact_registry_cache_fname):
            with open(artifact_registry_cache_fname, 'rb') as f:
                RT.artifact_registry_cache = pickle.load(f)
                assert isinstance(RT.artifact_registry_cache, dict)
                LOG(f'Artifact registry cache loaded from "{artifact_registry_cache_fname}"')
        
            kwargs['cache'] = RT.artifact_registry_cache
        else:
            LOG(f'Artifact registry cache "{artifact_registry_cache_fname}" is absent, nothing to load')
        
        RT.artifact_registry = S3ArtifactRegistry(maven_group_id=CONFIG.model_group_uri, **kwargs)
        LOG(f'S3ArtifactRegistry created')
    case _:
        assert False, f'Unsupported {artifact_registry_type=}'

if lc.version != 0:
    assert CONFIG.exec_mode != ExecMode.MASTER_NOTEBOOK, 'With MASTER_NOTEBOOK exec_mode one should not overwrite any of the launches (experiments)'
    RT.artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    meta = dict(
        hypers=HP._asdict(), 
        config=CONFIG._asdict(), 
    )
    
    with io.StringIO() as b:
        json.dump(meta, b)
        RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)
else:
    assert CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK, 'MASTER_NOTEBOOK exec_mode is for working on 0 (dummy) version only'

RT.optuna_trial = None
optuna_trial_fname = os.path.join(CONFIG.initrd_path, 'optuna_trial.json')

if os.path.exists(optuna_trial_fname):
    with open(os.path.join(optuna_trial_fname), 'rt') as f:
        RT.optuna_trial = json.load(f)
        assert 'trial_number' in RT.optuna_trial, RT.optuna_trial
        assert 'study_serial' in RT.optuna_trial, RT.optuna_trial
        assert 'study_name' in RT.optuna_trial, RT.optuna_trial

    LOG(f'Optuna trial loaded from "{optuna_trial_fname}": {RT.optuna_trial}')

summary_log_dir = lc.name

if RT.optuna_trial is not None:
    summary_log_dir = os.path.join(summary_log_dir, f'opt_{RT.optuna_trial['study_serial']}')
    
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')

summary_writer_type = os.environ.get('SUMMARY_WRITER', '').upper()

if not summary_writer_type:
    summary_writer_type = lu.when(CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK, 'S3', 'RMQ')

match summary_writer_type:
    case 'RMQ':
        kwargs = {}

        if os.environ.get('RMQ_CONNECTION_URL', None) is not None:
            kwargs['rmq_connection_url'] = os.environ['RMQ_CONNECTION_URL']

        RT.summary_writer = RmqSummaryWriter(log_dir=summary_log_dir, **kwargs)
        LOG(f'RmqSummaryWriter created')
    case 'S3':
        RT.summary_writer = S3SummaryWriter(log_dir=summary_log_dir)
        LOG(f'S3SummaryWriter created')
    case _:
        assert False, f'Unsupported {summary_writer_type=}'

RT.summary_writer.add_text('hyperparameters', pprint.pformat(HP._asdict(), sort_dicts=False), 0)
RT.summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 0)
RT.summary_writer.flush()

Random seed=1
torch.backends.cudnn.deterministic=True
ArtifactRegistry created
Tensorboard run=18e_action_plan_01/0
RmqSummaryWriter created


# Dataset

## Configure

In [45]:
# @launchit.disable
# @launchit.collect
HP.dataset.train = [
    '18b_dataset_04:1:cls=train,no=1,seq=8,mp=0,sh=3,210,160',
    '18b_dataset_04:1:cls=train,no=2,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=3,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=4,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=5,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=6,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=7,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=8,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=9,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=10,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=11,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=12,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=13,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=14,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=15,seq=8,mp=0,sh=3,210,160',
    # '18b_dataset_04:1:cls=train,no=16,seq=8,mp=0,sh=3,210,160',
]
HP.dataset.test = '18b_dataset_04:1:cls=test,no=1,seq=8,mp=0,sh=3,210,160'
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': None,
 'launch_id': 0,
 'general': {'comment': None, 'random_seed': 1, 'is_torch_deterministic': True},
 'dataset': {'train': ['18b_dataset_04:1:cls=train,no=1,seq=8,mp=0,sh=3,210,160',
                       '18b_dataset_04:1:cls=train,no=2,seq=8,mp=0,sh=3,210,160'],
             'test': '18b_dataset_04:1:cls=test,no=1,seq=8,mp=0,sh=3,210,160'},
 'model': {'parent': None,
           'actions_count': 6,
           'action_plan_length': 10,
           'd_action_plan': 3,
           'fsq': None,
           'encoder': None,
           'decoder': None,
           'is_trainable': True,
           'is_torch_compile': True},
 'train': {'epochs_count': 1,
           'batch_size': 32,
           'optimizer': 'AdamW',
           'learn_rate': 'const(0.00025)'}}


## Create

In [46]:
DATASET_NOTEBOOK_FNAME = '18b_dataset_04.ipynb'
source_code = launchit.extract_source_code(DATASET_NOTEBOOK_FNAME, collect_inds=[None, 'export_dataset'])
dataset_module = lu.make_module(f'dataset_notebook', source_code)
dataset_module.CONFIG = CONFIG
dataset_module.RNG = RNG
dataset_module.LOG = LOG
sys.modules['__main__'].Dataset = dataset_module.Dataset # Req-d for unpickling

In [47]:
import tempfile, shutil

def download_dataset(dataset_id):
    coords = hp_parse_artifact_source(dataset_id)
    classifier_suffix = lu.when(coords.asset_classifier, lambda: '.' + coords.asset_classifier, '')
    downloaded_pkl_gz_fname = os.path.join(CONFIG.private_data_path, f'{coords.model_name}.{coords.model_version}{classifier_suffix}.pkl.gz')
    
    if not os.path.exists(downloaded_pkl_gz_fname):
        pkl_gz = RT.artifact_registry.get_asset_content(
            coords.model_name, 
            coords.model_version, 
            asset_ext='gz', 
            asset_classifier=coords.asset_classifier, 
            maven_group_id=coords.group_id,
        )

        with open(downloaded_pkl_gz_fname, 'wb') as f:
            f.write(pkl_gz)
            LOG(f'"{dataset_id}" saved to "{downloaded_pkl_gz_fname}"')
    else:
        LOG(f'"{dataset_id}" is backed by "{downloaded_pkl_gz_fname}"')

    return downloaded_pkl_gz_fname

def load_dataset(fname):
    with gzip.GzipFile(filename=fname, mode='rb') as gzip_file:
        dataset = pickle.load(gzip_file)
        assert dataset.obs.shape[-3:] == (3, 210, 160), f'Unsupported {dataset.obs.shape[-3:]=}'
        return dataset

In [48]:
dataset_sequence_length = None

for dataset_name, source in zip(('train_dataset', 'test_dataset'), (HP.dataset.train, [HP.dataset.test])):
    dataset_action_plans = []
    
    for dataset_id in source:
        fname = download_dataset(dataset_id)
        dataset = load_dataset(fname)
    
        if dataset_sequence_length is None:
            dataset_sequence_length = dataset.sequence_length
        else:
            assert dataset.sequence_length == dataset_sequence_length
    
        dataset.init_iteration(batch_size=1024, is_shuffled=False)
        included_action_plans_count = 0
        
        for batch in dataset:
            action_plans = batch.actions.view(-1, batch.actions.shape[-1])
            pmasks = batch.pmasks.view(-1, batch.pmasks.shape[-1])
            is_full_sequence = pmasks.sum(dim=-1) == 0
            dataset_action_plans.append(action_plans[is_full_sequence])
            included_action_plans_count += is_full_sequence.sum().item()
    
        LOG(f'"{dataset_id}" processed: {included_action_plans_count} items')

    dataset_action_plans = torch.cat(dataset_action_plans, dim=0).to(CONFIG.cuda_device)
    tensor_dataset = TensorDataset(dataset_action_plans)
    setattr(RT, dataset_name, tensor_dataset)
    LOG(f'RT.{dataset_name}.action_plans.shape={tensor_dataset.tensors[0].shape}')

"18b_dataset_04:1:cls=train,no=1,seq=8,mp=0,sh=3,210,160" is backed by "/home/misha/dev/mine/neurolab/data/18_rl/18b_dataset_04.1.train,no=1,seq=8,mp=0,sh=3,210,160.pkl.gz"
"18b_dataset_04:1:cls=train,no=1,seq=8,mp=0,sh=3,210,160" processed: 14571 items
"18b_dataset_04:1:cls=train,no=2,seq=8,mp=0,sh=3,210,160" is backed by "/home/misha/dev/mine/neurolab/data/18_rl/18b_dataset_04.1.train,no=2,seq=8,mp=0,sh=3,210,160.pkl.gz"
"18b_dataset_04:1:cls=train,no=2,seq=8,mp=0,sh=3,210,160" processed: 14710 items
RT.train_dataset.action_plans.shape=torch.Size([29281, 8])
"18b_dataset_04:1:cls=test,no=1,seq=8,mp=0,sh=3,210,160" is backed by "/home/misha/dev/mine/neurolab/data/18_rl/18b_dataset_04.1.test,no=1,seq=8,mp=0,sh=3,210,160.pkl.gz"
"18b_dataset_04:1:cls=test,no=1,seq=8,mp=0,sh=3,210,160" processed: 14695 items
RT.test_dataset.action_plans.shape=torch.Size([14695, 8])


# Model

In [49]:
# @launchit.collect_export
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as VF
from torch.nn.attention import SDPBackend, sdpa_kernel
import torch_utils

## FSQ

In [90]:
# @launchit.collect_export
class FSQ(nn.Module):
    """
    Как это работает:
    1. Сжимаем вектор. Энкодер выдаёт вектор x длины всего 3 числа, например [0.82, -0.45, 0.12].
    2. Ограничиваем диапазон. Прогоняем его через tanh(e), чтобы все 3 числа гарантированно лежали в диапазоне от -1 до 1.
    3. Округляем до сетки (Квантование). Договариваемся, что каждое число может принимать только 4 значения: [-1.0, -0.33, 0.33, 1.0] (0, 1, 2, 3).
    	* 0.82 ближе всего к 1.0 (это уровень 3).
    	* -0.45 ближе всего к -0.33 (это уровень 1).
    	* 0.12 ближе всего к 0.33 (это уровень 2).
    	* Получаем квантованный вектор z = [1.0, -0.33, 0.33], который и идёт в Декодер.
    4. Переводим в один индекс (для Policy Head). Поскольку у нас 3 координаты и у каждой по 4 уровня (от 0 до 3), 
    мы просто переводим тройку уровней (3, 1, 2) из четверичной системы счисления в обычное число:
    index = 3×1+1×4+2×16 = 39
    Всего комбинаций: 4 × 4 × 4 = 64 индекса (от 0 до 63).
    """
    @dataclass(slots=True)
    class Params:
        levels: list = None # e.g. [4,4,4] -> 64 chords
        
    def __init__(self, params): 
        super().__init__()
        self.params = params
        self.register_buffer('levels', torch.tensor(params.levels, dtype=torch.long))
        assert torch.all(self.levels % 2 == 0), f'For now works (tested) for even levels only'
        self.register_buffer('half_levels', (self.levels - 1) / 2.0) # [4,4,4] -> [1.5, 1.5, 1.5]
        basis = torch.cumprod(torch.tensor([1] + self.levels.tolist()[:-1]), dim=0).to(self.levels.dtype) # [1, 4, 16]
        self.register_buffer('basis', basis)

    def __repr__(self):
        params = []

        for k, v in dataclasses.asdict(self.params).items():
            params.append(f'{k}={v}')
            
        return (
            f'{self.__class__.__name__}' + 
            '(' + 
            ', '.join(params) +
            ', ' + f'capacity={self.capacity}' +
            ')'
        )

    ForwardResult = namedtuple('ForwardResult', 'q indices')

    def forward(self, x):
        # x (action plan encodings): (..., levels)
        assert x.shape[-1] == len(self.levels)
        y = torch.tanh(x) # clamp (with no grads lost) x to [-1, +1]
        y_scaled = (y + 1.0) * self.half_levels # [-1, +1] -> [0, levels-1]
        y_rounded = torch.round(y_scaled) # grads is lost here
        y_discrete = y_scaled + (y_rounded - y_scaled).detach() # Straight-Through Estimator lets grads to survive round
        q = (y_discrete / self.half_levels) - 1.0 # back to [-1, +1]
        indices = (y_rounded.long() * self.basis).sum(dim=-1)
        return self.ForwardResult(
            q=q, 
            indices=indices,
        )

    def qtify(self, indices):
        # indices of action plans (chords): any shape
        coords = (indices.unsqueeze(-1).long() // self.basis) % self.levels
        return (coords.float() / self.half_levels) - 1.0

    @property
    def capacity(self):
        return math.prod(self.params.levels)        

### Test: basics

In [91]:
# @launchit.disable
t = lu.ScopedVars()
t.fsq = FSQ(FSQ.Params(
    levels=[4, 4, 4],
))
print(t.fsq)

t.test_cases = [
    dict(
        x=[0.82, -0.45, 0.12], 
        q=[1.0, -1/3, 1/3], 
        indices=[3*1 + 1*4 + 2*16]
    ),
    dict(
        x=[-100.0, -100.0, -100.0], 
        q=[-1, -1, -1], 
        indices=[0*1 + 0*4 + 0*16]
    ),
    dict(
        x=[100.0, 100.0, 100.0], 
        q=[1, 1, 1], 
        indices=[3*1 + 3*4 + 3*16]
    ),
    dict(
        x=[0.0, 0.0, 0.0], 
        q=[1/3, 1/3, 1/3], 
        indices=[2*1 + 2*4 + 2*16]
    ),
    dict(
        x=[[0.0, 0.0, 0.0], [-1, 0.0, 1.0]], 
        q=[[1/3, 1/3, 1/3], [-1, 1/3, 1]], 
        indices=[2*1 + 2*4 + 2*16, 0*1 + 2*4 + 3*16]
    ),
]

for t.test_case_ind, t.test_case in enumerate(t.test_cases):
    t.q, t.indices = t.fsq(torch.tensor(t.test_case['x']))
    t.true_q = torch.tensor(t.test_case['q']).float()
    assert torch.all(torch.isclose(t.q, t.true_q)), f'Failed on {t.test_case_ind=}'
    t.true_indices = torch.tensor(t.test_case['indices'])
    assert torch.all(t.indices == t.true_indices), f'Failed on {t.test_case_ind=}'

    t.q = t.fsq.qtify(t.true_indices)
    assert torch.all(torch.isclose(t.q, t.true_q)), f'Failed on {t.test_case_ind=}'

FSQ(levels=[4, 4, 4], capacity=64)


## ActionPlanEncoder

In [52]:
# @launchit.collect_export
class ActionPlanEncoder(torch_utils.TrainableModule):
    @dataclass(slots=True)
    class Params:
        actions_count: int = 6
        action_plan_length: int = 10
        d_action_plan: int = 3
        features_counts: tuple = (32, 64, 64)

    def __init__(self, params):
        super().__init__()
        self.params = params
        assert len(self.params.features_counts) == 3
        fcs = self.params.features_counts
        self.action_encoding = nn.Embedding(self.params.actions_count, fcs[0])
        self.conv = nn.Sequential(
            nn.Conv1d(fcs[0], fcs[1], kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(fcs[1], fcs[2], kernel_size=3, padding=1),
            nn.GELU(),
            nn.Flatten(),
        )
        self.fc = nn.Linear(fcs[-1] * self.params.action_plan_length, self.params.d_action_plan)

    def __repr__(self):
        params = []

        for k, v in dataclasses.asdict(self.params).items():
            params.append(f'{k}={v}')
            
        return (
            f'{self.__class__.__name__}' + 
            '(' + 
            ', '.join(params) +
            ')'
        )
        
    def forward(self, action_plans):
        # action_plans: (..., plan_len)
        action_plans_shape = action_plans.shape
        action_plans = action_plans.view(-1, action_plans.shape[-1]) # (uberbatch, plan_len)
        assert action_plans.shape[-1] == self.params.action_plan_length
        x = self.action_encoding(action_plans) # (uberbatch, plan_len, features_counts[0])
        x = x.transpose(1, 2) # (uberbatch, features_counts[0], plan_len)
        x = self.conv(x) # (uberbatch, features_counts[-1], plan_len)
        x = self.fc(x) # (uberbatch, d_action_plan)
        return x.view(*action_plans.shape[:-1], x.shape[-1]) # (..., d_action_plan)

    @property
    def grad_norm_groups(self):
        groups = dict(
            action_encoding=list(self.action_encoding.parameters()),
            conv=list(self.conv.parameters()),
            fc=list(self.fc.parameters()),
        )

        return groups

### Test: basics

In [53]:
# @launchit.disable
t = lu.ScopedVars()
t.ape_params = ActionPlanEncoder.Params(
    actions_count=6,
    action_plan_length=10,
    d_action_plan=3,
    features_counts=(32, 64, 64),
)
t.encoder = ActionPlanEncoder(t.ape_params)
print(t.encoder)
print(nn.Module.__repr__(t.encoder))
t.envs_count = 2
t.action_plans = torch.randint(0, t.ape_params.actions_count, size=(t.envs_count, t.ape_params.action_plan_length))
print(f'{t.action_plans=}')
t.action_plan_encodings = t.encoder(t.action_plans)
print(f'{t.action_plan_encodings=}')
assert t.action_plan_encodings.shape == (t.envs_count, t.ape_params.d_action_plan)

ActionPlanEncoder(actions_count=6, action_plan_length=10, d_action_plan=3, features_counts=(32, 64, 64))
ActionPlanEncoder(
  (action_encoding): Embedding(6, 32)
  (conv): Sequential(
    (0): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): GELU(approximate='none')
    (2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (3): GELU(approximate='none')
    (4): Flatten(start_dim=1, end_dim=-1)
  )
  (fc): Linear(in_features=640, out_features=3, bias=True)
)
t.action_plans=tensor([[1, 2, 0, 3, 2, 0, 2, 3, 0, 2],
        [4, 1, 5, 1, 1, 2, 5, 0, 2, 4]])
t.action_plan_encodings=tensor([[-0.0587,  0.0191,  0.0214],
        [-0.0147,  0.0829,  0.0525]], grad_fn=<ViewBackward0>)


### Test: grad_norm_groups

In [54]:
# @launchit.disable
t = lu.ScopedVars()
t.ape_params = ActionPlanEncoder.Params(
    actions_count=6,
    action_plan_length=10,
    d_action_plan=3,
    features_counts=(32, 64, 64),
)
t.encoder = ActionPlanEncoder(t.ape_params)
print(list(t.encoder.grad_norm_groups.keys()))

for t.n, t.params in t.encoder.grad_norm_groups.items():
    for t.p_ind, t.p in enumerate(t.params):
        assert isinstance(t.p, torch.nn.parameter.Parameter), (t.n, t.p_ind)

['action_encoding', 'conv', 'fc']


## ActionPlanDecoder

In [55]:
# @launchit.collect_export
class ActionPlanDecoder(torch_utils.TrainableModule):
    @dataclass(slots=True)
    class Params:
        actions_count: int = 6
        action_plan_length: int = 10
        d_action_plan: int = 3
        d_model: int = 256 # None if no state conditioning
        d_inner: int = 256
        fsq: dict = None

    def __init__(self, params):
        super().__init__()
        self.params = params
        self.mlp = nn.Sequential(
            nn.Linear(self.params.d_action_plan + lu.coalesce(self.params.d_model, 0), self.params.d_inner),
            nn.LayerNorm(self.params.d_inner),
            nn.GELU(),
            nn.Linear(self.params.d_inner, self.params.d_inner),
            nn.GELU(),
            nn.Linear(self.params.d_inner, self.params.action_plan_length * self.params.actions_count),
        )
        self.fsq = FSQ(FSQ.Params(**self.params.fsq))

    def __repr__(self):
        params = []

        for k, v in dataclasses.asdict(self.params).items():
            params.append(f'{k}={v}')
            
        return (
            f'{self.__class__.__name__}' + 
            '(' + 
            ', '.join(params) +
            ')'
        )

    ForwardResult = namedtuple('ForwardResult', 'action_plan_logits action_plans')
    
    def forward(self, action_plan_qencodings=None, action_plan_inds=None, states=None):
        # action_plan_qencodings: (..., d_action_plan)
        # action_plan_inds: (...)
        # states (None if no state conditioning): (..., d_model)
        assert (action_plan_qencodings is not None) ^ (action_plan_inds is not None), f'Either action_plan_qencodings or action_plan_inds must be specified (mutually exclusive)'

        if action_plan_inds is not None:
            action_plan_qencodings = self.fsq.qtify(action_plan_inds)
        
        layout = action_plan_qencodings.shape[:-1]
        assert (states is None) == (self.params.d_model is None)
          
        if states is not None:
            assert layout == states.shape[:-1]
            x = torch.cat([action_plan_qencodings, states], dim=-1)
        else:
            x = action_plan_qencodings

        x = self.mlp(x)
        x = x.view(*layout, self.params.action_plan_length, self.params.actions_count)  # (..., plan_len, actions)
        return self.ForwardResult(
            action_plan_logits=x,
            action_plans=x.argmax(dim=-1),
        )

    @property
    def grad_norm_groups(self):
        groups = dict(
            mlp=list(self.mlp.parameters()),
        )

        return groups

### Test: basics

In [56]:
# @launchit.disable
t = lu.ScopedVars()
t.fsq = FSQ(FSQ.Params(
    levels=[4, 4, 4],
))
t.apd_params = ActionPlanDecoder.Params(
    actions_count=6,
    action_plan_length=10,
    d_action_plan=3,
    d_model=None,
    d_inner=256,
    fsq=dataclasses.asdict(t.fsq.params),
)
t.decoder = ActionPlanDecoder(t.apd_params)
print(t.decoder)
print(nn.Module.__repr__(t.decoder))
t.envs_count = 2
t.action_plan_encodings = torch.randn(t.envs_count, t.apd_params.d_action_plan)
t.action_plan_qencodings = t.fsq(t.action_plan_encodings).q
print(f'{t.action_plan_qencodings=}')
t.fr = t.decoder(t.action_plan_qencodings)
print(f'{t.fr.action_plans=}')
assert t.fr.action_plans.shape == (t.envs_count, t.apd_params.action_plan_length)

t.action_plan_inds = torch.randint(0, math.prod(t.fsq.params.levels), size=(t.envs_count,))
print(f'{t.action_plan_inds=}')
t.fr = t.decoder(action_plan_inds=t.action_plan_inds)
print(f'{t.fr.action_plans=}')
assert t.fr.action_plans.shape == (t.envs_count, t.apd_params.action_plan_length)

ActionPlanDecoder(actions_count=6, action_plan_length=10, d_action_plan=3, d_model=None, d_inner=256, fsq={'levels': [4, 4, 4]})
ActionPlanDecoder(
  (mlp): Sequential(
    (0): Linear(in_features=3, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (2): GELU(approximate='none')
    (3): Linear(in_features=256, out_features=256, bias=True)
    (4): GELU(approximate='none')
    (5): Linear(in_features=256, out_features=60, bias=True)
  )
  (fsq): FSQ(levels=[4, 4, 4])
)
t.action_plan_qencodings=tensor([[ 0.3333, -1.0000,  0.3333],
        [-0.3333,  1.0000,  0.3333]])
t.fr.action_plans=tensor([[1, 5, 2, 1, 3, 1, 2, 4, 2, 1],
        [5, 4, 0, 1, 3, 2, 2, 1, 1, 0]])
t.action_plan_inds=tensor([23, 12])
t.fr.action_plans=tensor([[5, 5, 5, 1, 2, 1, 2, 5, 1, 1],
        [2, 1, 5, 1, 3, 3, 2, 5, 2, 1]])


### Test: grad_norm_groups

In [57]:
# @launchit.disable
t = lu.ScopedVars()
t.fsq = FSQ(FSQ.Params(
    levels=[4, 4, 4],
))
t.apd_params = ActionPlanDecoder.Params(
    actions_count=6,
    action_plan_length=10,
    d_action_plan=3,
    d_model=None,
    d_inner=256,
    fsq=dataclasses.asdict(t.fsq.params),
)
t.decoder = ActionPlanDecoder(t.apd_params)
print(list(t.decoder.grad_norm_groups.keys()))

for t.n, t.params in t.decoder.grad_norm_groups.items():
    for t.p_ind, t.p in enumerate(t.params):
        assert isinstance(t.p, torch.nn.parameter.Parameter), (t.n, t.p_ind)

['mlp']


## Configure

In [67]:
# @launchit.disable
# @launchit.collect
HP.model.parent = None
HP.model.actions_count = 6
HP.model.action_plan_length = 8
HP.model.d_action_plan = 3
HP.model.fsq = dict(levels=[4, 4, 4])
HP.model.encoder = dict(features_counts=(32, 64, 64))
HP.model.decoder = dict(d_inner=256)
HP.model.is_torch_compile = True
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': None,
 'launch_id': 0,
 'general': {'comment': None, 'random_seed': 1, 'is_torch_deterministic': True},
 'dataset': {'train': ['18b_dataset_04:1:cls=train,no=1,seq=8,mp=0,sh=3,210,160',
                       '18b_dataset_04:1:cls=train,no=2,seq=8,mp=0,sh=3,210,160'],
             'test': '18b_dataset_04:1:cls=test,no=1,seq=8,mp=0,sh=3,210,160'},
 'model': {'parent': None,
           'actions_count': 6,
           'action_plan_length': 8,
           'd_action_plan': 3,
           'fsq': {'levels': [4, 4, 4]},
           'encoder': {'features_counts': (32, 64, 64)},
           'decoder': {'d_inner': 256},
           'is_trainable': True,
           'is_torch_compile': True},
 'train': {'epochs_count': 4,
           'batch_size': 128,
           'optimizer': 'AdamW',
           'learn_rate': 'const(0.0005)'}}


## Initrd

In [68]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig':
    if HP.model.parent is not None:
        for model_name in RT.model_names:
            model_parent = lu.when(
                isinstance(HP.model.parent, dict),
                lambda: HP.model.parent[model_name],
                HP.model.parent,
            )
        
            if model_parent is not None:
                coords = hp_parse_artifact_source(HP.model.parent)
                CONFIG.artifact_registry.get_asset_content(
                    coords.model_name, 
                    coords.model_version, 
                    asset_ext='json', 
                    asset_classifier=f'{model_name}_params', 
                    maven_group_id=coords.group_id,
                )
                CONFIG.artifact_registry.get_asset_content(
                    coords.model_name, 
                    coords.model_version, 
                    asset_ext='pt', 
                    asset_classifier=model_name, 
                    maven_group_id=coords.group_id,
                )
# @launchit.stop

## Create

In [69]:
fsq_params = FSQ.Params(
    **HP.model.fsq,
)
RT.fsq = FSQ(fsq_params)
LOG(f'{RT.fsq=} created')

ape_params = ActionPlanEncoder.Params(
    actions_count=HP.model.actions_count,
    action_plan_length=HP.model.action_plan_length,
    d_action_plan=HP.model.d_action_plan,
    **HP.model.encoder,
)
RT.encoder = ActionPlanEncoder(ape_params).to(CONFIG.cuda_device)
LOG(f'{RT.encoder=} created')

apd_params = ActionPlanDecoder.Params(
    actions_count=HP.model.actions_count,
    action_plan_length=HP.model.action_plan_length,
    d_action_plan=HP.model.d_action_plan,
    d_model=None,
    fsq=HP.model.fsq,
    **HP.model.decoder,
)
RT.decoder = ActionPlanDecoder(apd_params).to(CONFIG.cuda_device)
LOG(f'{RT.decoder=} created')

if HP.model.parent is not None:
    for model_name in RT.model_names:
        model_parent = lu.when(
            isinstance(HP.model.parent, dict),
            lambda: HP.model.parent[model_name],
            HP.model.parent,
        )
    
        if model_parent is not None:
            model = getattr(RT, model_name)
            coords = hp_parse_artifact_source(model_parent)
            model_weights_data = RT.artifact_registry.get_asset_content(
                coords.model_name, 
                coords.model_version, 
                asset_ext='pt', 
                asset_classifier=model_name, 
                maven_group_id=coords.group_id,
            )
            
            with io.BytesIO(model_weights_data) as b:
                kwargs = {}
                
                if not CONFIG.is_cuda:
                    kwargs['map_location'] = torch.device('cpu')
        
                state_dict = torch.load(b, **kwargs)
                model.load_state_dict(state_dict)
        
            LOG(f'RT.{model_name} state loaded from "{model_parent}"')

if HP.model.is_torch_compile is not None:
    for model_name in RT.model_names:
        model_is_torch_compile = lu.when(
            isinstance(HP.model.is_torch_compile, dict),
            lambda: HP.model.is_torch_compile[model_name],
            HP.model.is_torch_compile,
        )

        if model_is_torch_compile:
            model = getattr(RT, model_name)
            compiled_model = torch.compile(model, fullgraph=True)
            setattr(RT, model_name, compiled_model)
            LOG(f'RT.{model_name} compiled')


if HP.model.is_trainable is not None:
    for model_name in RT.model_names:
        model_is_trainable = lu.when(
            isinstance(HP.model.is_trainable, dict),
            lambda: HP.model.is_trainable[model_name],
            HP.model.is_trainable,
        )

        if model_is_trainable:
            model = getattr(RT, model_name)
            
            if hasattr(model, 'is_trainable'):
                LOG(f'RT.{model_name}.is_trainable={model.is_trainable}')
            else:
                LOG(f'RT.{model_name} has no "is_trainable" property')

RT.fsq=FSQ(levels=[4, 4, 4]) created
RT.encoder=ActionPlanEncoder(actions_count=6, action_plan_length=8, d_action_plan=3, features_counts=(32, 64, 64)) created
RT.decoder=ActionPlanDecoder(actions_count=6, action_plan_length=8, d_action_plan=3, d_model=None, d_inner=256, fsq={'levels': [4, 4, 4]}) created
RT.fsq compiled
RT.encoder compiled
RT.decoder compiled
RT.fsq has no "is_trainable" property
RT.encoder.is_trainable=True
RT.decoder.is_trainable=True


# TRAIN

## Configure

In [70]:
# @launchit.disable
# @launchit.collect
# Training params
HP.train.epochs_count = 4
HP.train.batch_size = 128 
HP.train.optimizer = 'AdamW'
HP.train.learn_rate = 'const(0.0005)'

# Reporting params
HP.report.follow_up_freq = 10
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': None,
 'launch_id': 0,
 'general': {'comment': None, 'random_seed': 1, 'is_torch_deterministic': True},
 'dataset': {'train': ['18b_dataset_04:1:cls=train,no=1,seq=8,mp=0,sh=3,210,160',
                       '18b_dataset_04:1:cls=train,no=2,seq=8,mp=0,sh=3,210,160'],
             'test': '18b_dataset_04:1:cls=test,no=1,seq=8,mp=0,sh=3,210,160'},
 'model': {'parent': None,
           'actions_count': 6,
           'action_plan_length': 8,
           'd_action_plan': 3,
           'fsq': {'levels': [4, 4, 4]},
           'encoder': {'features_counts': (32, 64, 64)},
           'decoder': {'d_inner': 256},
           'is_trainable': True,
           'is_torch_compile': True},
 'train': {'epochs_count': 4,
           'batch_size': 128,
           'optimizer': 'AdamW',
           'learn_rate': 'const(0.0005)'}}


## Create

In [71]:
ump = hp_parse_universal_module(HP.train.optimizer)
assert not ump.args
models = [getattr(RT, model_name) for model_name in RT.model_names]
all_params = list(itertools.chain.from_iterable(model.parameters() for model in models))
optimized_params = list(filter(lambda p: p.requires_grad, all_params))
optimizer = getattr(torch.optim, ump.module_name)(optimized_params, **ump.kwargs)
all_params_count = sum(p.numel() for p in all_params)
optimized_params_count = sum(p.numel() for p in optimized_params)
LOG(f'{all_params_count=:_}, {optimized_params_count=:_}')

ump = hp_parse_universal_module(HP.train.learn_rate)
lr_anneal = get_anneal(ump.module_name, *ump.args, **ump.kwargs)

all_params_count=99_955, optimized_params_count=99_955


In [72]:
train_data_loader = DataLoader(
    RT.train_dataset,
    batch_size=HP.train.batch_size,
    shuffle=True,     
    drop_last=False,
)
LOG(f'{len(train_data_loader)=}')

test_data_loader = DataLoader(
    RT.test_dataset,
    batch_size=HP.train.batch_size,
    shuffle=False,     
    drop_last=False,
)
LOG(f'{len(test_data_loader)=}')

In [ ]:
def create_is_follow_up(period=None):
    assert period is not None
    next_gate = 0
    
    def every_thunk(counter, is_last_step):
        nonlocal next_gate

        if counter >= next_gate:
            next_gate += period
            return True

        return is_last_step

    return every_thunk

is_follow_up = create_is_follow_up(HP.train.epochs_count // HP.report.follow_up_freq)

## Train

In [64]:
metrics_suite = defaultdict(list)
timings = {}

for epoch in tqdm(range(HP.train.epochs_count), desc='Epoch', disable=not CONFIG.is_interactive):
    timings.clear()
    t0 = time.monotonic()

    # EXECUTE EXTERNAL COMMANDS
    if command := RT.command_listener.get_command():
        LOG(f'Got {command=}')
        
        if command == 'stop':
            RT.should_save_model = True
            break
        elif command == 'abort':
            RT.should_save_model = False
            break
        else:
            LOG(f'Ignoring unknown {command=}')

    # ANNEAL
    progress = epoch / HP.train.epochs_count
    lr = lr_anneal(progress)
    for param_group in optimizer.param_groups: param_group['lr'] = lr

    # OPTIMIZATION
    t1 = time.monotonic()
    epoch_loss = RecursiveAverageFilter()

    RT.encoder.train()
    RT.decoder.train()

    for batch in tqdm(train_data_loader, desc='Train batch', disable=not CONFIG.is_interactive, leave=False):
        action_plans, = batch
        batch_size = len(action_plans)
        assert action_plans.shape == (batch_size, HP.model.action_plan_length)

        action_plan_encodings = RT.encoder(action_plans) 
        assert action_plan_encodings.shape == (batch_size, HP.model.action_plan_length, HP.model.d_action_plan)
        
        action_plan_qencodings = RT.fsq(action_plan_encodings)
        assert action_plan_qencodings.shape == (batch_size, HP.model.action_plan_length, HP.model.d_action_plan)
        
        action_plan_logits = RT.decoder(action_plan_qencodings=action_plan_qencodings).action_plan_logits
        assert assert.shape == (batch_size, HP.model.action_plan_length, HP.model.actions_count)

        loss = F.cross_entropy(action_plan_logits.view(-1, HP.model.actions_count), action_plans.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss(loss.item(), batch_size)
        processed_batch_items += batch_size

    LOG(f'Optimization done: loss={epoch_loss.v:.4f}, {len(RT.train_dataset):_} batch items processed', when=not CONFIG.is_interactive)
    timings['optimization'] = time.monotonic() - t1

    # FOLLOW-UP optimization
    if is_follow_up(epoch, is_last_step=(epoch + 1 >= HP.train.epochs_count)):
        t1 = time.monotonic()
        flat_matches, plan_matches, action_plan_inds = [], [], []
        
        RT.encoder.eval()
        RT.decoder.eval()
            
        with torch.no_grad():
            for batch in tqdm(test_data_loader, desc='Test batch', disable=not CONFIG.is_interactive, leave=False):
                action_plans, = batch
                batch_size = len(action_plans)
                assert action_plans.shape == (batch_size, HP.model.action_plan_length)
        
                action_plan_encodings = RT.encoder(action_plans) 
                assert action_plan_encodings.shape == (batch_size, HP.model.action_plan_length, HP.model.d_action_plan)
                
                action_plan_qencodings = RT.fsq(action_plan_encodings)
                assert action_plan_qencodings.shape == (batch_size, HP.model.action_plan_length, HP.model.d_action_plan)
                
                decoded_action_plans, decoded_action_plan_inds = RT.decoder(action_plan_qencodings=action_plan_qencodings)
                assert decoded_action_plans.shape == (batch_size, HP.model.action_plan_length)

                matches = action_plans.view(-1) == decoded_action_plans.view(-1)
                flat_matches.append(matches)
                
                matches = matches.view(-1, HP.model.action_plan_length)
                matches = matches.all(dim=-1) # find all rows full of True
                plan_matches.append(matches)

                action_plan_inds.append(decoded_action_plan_inds)

        flat_matches = torch.cat(flat_matches, dim=0)
        assert flat_matches.ndim == 1
        flat_accuracy = flat_matches.mean().item()

        plan_matches = torch.cat(plan_matches, dim=0)
        assert plan_matches.ndim == 1
        plan_accuracy = plan_matches.mean().item()

        action_plan_inds = torch.cat(action_plan_inds, dim=0)
        assert action_plan_inds.ndim == 1
        unique_action_plan_inds = torch.unique(action_plan_inds).item()
        capacity_utilization = unique_action_plan_inds / RT.fsq.capacity
        
        RT.summary_writer.add_scalar('test/a) flat_accuracy', flat_accuracy, epoch)
        metrics_suite['flat_accuracy'].append(flat_accuracy)
        RT.summary_writer.add_scalar('test/b) plan_accuracy', plan_accuracy, epoch)
        metrics_suite['plan_accuracy'].append(plan_accuracy)
        RT.summary_writer.add_scalar('test/c) capacity_utilization', capacity_utilization, epoch)
        RT.summary_writer.add_scalar('test/d) uniques', unique_action_plan_inds, epoch)

        LOG(f'Test done: {flat_accuracy=:.4f}, {plan_accuracy=:.4f}', when=not CONFIG.is_interactive)
        timings['test'] = time.monotonic() - t1
        t1 = time.monotonic()

    # REPORT
    t_end = time.monotonic()
    RT.summary_writer.add_scalar('timings/a) eps', 1 / (t_end - t0), epoch) # epochs per second, bigger = better
    RT.summary_writer.add_scalar('timings/b) bips', len(RT.train_dataset) / (t_end - t0), epoch) # batch items per second, bigger = better
    RT.summary_writer.add_scalar('timings/c) total', t_end - t0, epoch) # time per whole cycle, lesset = better
    RT.summary_writer.add_scalar('timings/d) optimization', timings['optimization'], epoch) # time per optimization, lesser = better

    if 'test' in timings:
        RT.summary_writer.add_scalar('timings/e) test', timings['test'], epoch) 
    
    RT.summary_writer.add_scalar('anneal/a) learn_rate', lr, epoch)
    
    RT.summary_writer.add_scalar('optimization/a) loss', epoch_loss.reset(), epoch)
    
    RT.summary_writer.flush()

    progress = (epoch + 1) / HP.train.epochs_count
    LOG(f'Progress {100 * progress:.2f}% ({epoch + 1:_} / {HP.train.epochs_count:_})', when=not CONFIG.is_interactive)

Epoch:   0%|          | 0/4 [00:00<?, ?it/s]

Train batch:   0%|          | 0/120 [00:00<?, ?it/s]

Test batch:   0%|          | 0/120 [00:00<?, ?it/s]

Train batch:   0%|          | 0/120 [00:00<?, ?it/s]

Test batch:   0%|          | 0/120 [00:00<?, ?it/s]

Train batch:   0%|          | 0/120 [00:00<?, ?it/s]

Test batch:   0%|          | 0/120 [00:00<?, ?it/s]

Train batch:   0%|          | 0/120 [00:00<?, ?it/s]

Test batch:   0%|          | 0/120 [00:00<?, ?it/s]

## Save

In [ ]:
# @launchit.disable_worker
lc = HP.launch_component()

if lc.version != 0:
    if not RT.should_save_model:
        LOG(f'Model saving skipped')
    else:
        for model_name in RT.model_names:
            model = getattr(RT, model_name)
            
            with io.BytesIO() as b:
                # Force both compiled and non-compiled models to be saved with the same keys
                state_dict = lu.when(hasattr(model, '_orig_mod'), lambda: model._orig_mod.state_dict(), lambda: model.state_dict())
                torch.save(state_dict, b)
                RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier=model_name, replace=True)

            if hasattr(model, 'params'):
                with io.StringIO() as b:
                    json.dump(dataclasses.asdict(model.params), b)
                    RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier=f'{model_name}_params', replace=True)

            LOG(f'RT.{model_name} saved to artifact registry')
    
        with io.StringIO() as b:
            json.dump(metrics_suite, b)
            RT.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='metrics_suite', replace=True)
            LOG(f'Metrics suite saved to artifact registry')
    
        with open(CONFIG.metrics_suite_fname, 'w') as f:
            json.dump(metrics_suite, f)
            LOG(f'Metrics suite saved to "{CONFIG.metrics_suite_fname}"')

    if LOG.log_fname is not None and os.path.exists(LOG.log_fname):
        import gzip, shutil
        gzip_fname = LOG.log_fname + '.gz'

        with open(LOG.log_fname, 'rb') as log_file:
            with gzip.open(gzip_fname, 'wb') as gzip_file:
                shutil.copyfileobj(log_file, gzip_file)
        
        RT.summary_writer.add_file(gzip_fname, os.path.basename(gzip_fname))
        RT.summary_writer.flush()
        LOG(f'Log file "{LOG.log_fname}" captured')

# LaunchIt!

## LAUNCH_NOTEBOOK

In [105]:
# @launchit.disable
launchit_t0 = time.monotonic()

In [107]:
# @launchit.disable
launchit_interval = time.monotonic() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    RT.artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
    )
    launch_notebook_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=component_version, 
        expandvars=expandvars, 
        collect_inds=[], 
        disable_inds=[]
    )
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=143
Creating /home/misha/dev/mine/neurolab/18_rl/18d_world_model_09-launch143.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/18_rl/18d_world_model_09-launch143.ipynb"


## DOCKER_LAUNCH_NOTEBOOK

In [ ]:
# @launchit.disable
launchit_t0 = time.monotonic()

In [ ]:
# @launchit.collect_manual_run_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    import launch_dispatcher
    image_tag = os.path.join(CONFIG.docker_registry, '${MODEL_NAME}' + ':' + '${MODEL_VERSION}')
    launch_request = dict(
        launch_image=image_tag,
        keep_container=False,
    )
    launch_dispatcher.LaunchRequest.run(launch_request)
# @launchit.stop

In [ ]:
# @launchit.disable
launchit_interval = time.monotonic() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    RT.artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
    )
    launch_notebook_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=component_version, 
        expandvars=expandvars, 
        collect_inds=['initrd', 'build_docker_launch', 'manual_run_docker_launch', 'temp_config'], 
        disable_inds=[]
    )
    LOG(f'Created docker launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

# System

## Create optuna trial

In [ ]:
# @launchit.collect_optuna
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK and os.path.exists('${OPTUNA_STUDY_FNAME}'):
    import re
    import launchit
    optuna_study_fname = '${OPTUNA_STUDY_FNAME}'
    optuna_study_storage = JournalStorage(JournalFileBackend(optuna_study_fname))
    optuna_study_name = '${OPTUNA_STUDY_NAME}'
    assert optuna_study_name
    assert optuna_study_name != '$' + '{OPTUNA_STUDY_NAME}', optuna_study_name
    
    optuna_study_nb_fname = re.sub('.optuna$', '.ipynb', optuna_study_fname)
    source_code = launchit.extract_source_code(optuna_study_nb_fname)
    Logging.get()(f'Extracted source code for set_hyperparamters from "{optuna_study_nb_fname}"')
    module = lu.make_module('optuna_study_hyperparameters', source_code)
    grid_search_space = lu.when(hasattr(module, 'GRID_SEARCH_SPACE'), lambda: module.GRID_SEARCH_SPACE, None)
    
    optuna_study = optuna.load_study(
        study_name=optuna_study_name, 
        storage=optuna_study_storage,
        sampler=lu.when(grid_search_space, lambda: optuna.samplers.GridSampler(grid_search_space), None)
    )
    
    optuna_trial = optuna_study.ask()
    optuna_trial.set_user_attr('MODEL_VERSION', '${MODEL_VERSION}')
    optuna_study_serial = optuna_study.user_attrs['STUDY_SERIAL']
    
    HP = module.set_hyperparameters(Hyperparameters(), optuna_study, optuna_trial)
    Logging.get()(f'HP=\n{pprint.pformat(HP._asdict(), sort_dicts=False)}\n')

    assert os.path.exists(CONFIG.initrd_path)

    with open(os.path.join(CONFIG.initrd_path, 'optuna_trial.json'), 'w') as f:
        optuna_trial_dict = dict(
            trial_number=optuna_trial.number,
            study_serial=optuna_study_serial,
            study_name=optuna_study_name,
        )
        json.dump(optuna_trial_dict, f)

    with open(os.path.join(CONFIG.initrd_path, 'hyperparameters.json'), 'w') as f:
        json.dump(HP._asdict(), f)
        
elif CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK:
    hyperparameters_fname = os.path.join(CONFIG.initrd_path, 'hyperparameters.json')
    
    with open(hyperparameters_fname, 'r') as f:
        HP = Hyperparameters.from_dict(json.load(f))
        Logging.get()(f'HP loaded from "{hyperparameters_fname}"')
        Logging.get()(f'HP=\n{pprint.pformat(HP._asdict(), sort_dicts=False)}\n')


## Run optuna trial

In [ ]:
# @launchit.collect_optuna_run_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    # optuna_study and optuna_trial are created in cell above
    assert optuna_study is not None
    assert optuna_trial is not None

    import launch_dispatcher
    short_image_tag = '${MODEL_NAME}' + ':' + '${MODEL_VERSION}'
    image_tag = os.path.join(CONFIG.docker_registry, short_image_tag)
    launch_request = dict(
        launch_image=image_tag,
        result_fname=os.path.join(project_root_path, CONFIG.relative_metrics_suite_fname),
        keep_container=False,
    )
    launch_result_metadata, launch_result_body = launch_dispatcher.LaunchRequest.run(launch_request)
    
    with open(CONFIG.self_fname + '.out', mode='wt') as out_file:
        if launch_result_metadata['is_ok']:
            if launch_result_body:
                decisive_metric = '${OPTUNA_DECISIVE_METRIC}'
                
                with io.BytesIO(launch_result_body) as b:
                    launch_result_dict = json.load(b)
    
                if launch_result_dict.get(decisive_metric, []):
                    scalar_result = launch_result_dict[decisive_metric][-1]
                    optuna_study.tell(optuna_trial, scalar_result, state=optuna.trial.TrialState.COMPLETE)
    
                    message = f'Trial {optuna_trial.number} ({short_image_tag}) finished with value: {scalar_result} and parameters: {optuna_trial.params}'
                    
                    try:
                        best_trial = optuna_study._get_best_trial(deepcopy=False)
                        message += f'. Best is trial {best_trial.number} with value: {best_trial.value}'
                    except ValueError:
                        # If no feasible trials are completed yet, study.best_trial raises ValueError
                        pass

                    out_file.write(message)
                else:
                    optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
                    out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) finished with empty/missing "{decisive_metric}"')
            else:
                optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
                out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) finished with empty result body: {launch_result_metadata=}')
        else:
            optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
            out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) failed: {launch_result_metadata=}')

    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, message="To exit: use 'exit'")
    sys.exit(0)

## Save artifact registry cache

In [ ]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig':
    if CONFIG.artifact_registry_cache:
        with open(CONFIG.artifact_registry_cache_fname, 'wb') as f:
            pickle.dump(CONFIG.artifact_registry_cache, f)

        Logging.get()(f'Artifact registry cache saved to "{CONFIG.artifact_registry_cache_fname}"')
    else:
        Logging.get()(f'Artifact registry cache is empty, nothing to save')
# @launchit.stop

## Docker build

In [ ]:
# @launchit.collect_build_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    import subprocess
    import tempfile

    assert os.path.exists(os.path.join(build_project_root_path, CONFIG.relative_self_fname))

    dockerfile_content = f'''
FROM {os.path.join(CONFIG.docker_registry, '18x_data:latest')}
USER 1000
WORKDIR /neurolab
RUN --mount=type=secret,id=neurolab_deploy_key,target=/tmp/neurolab_deploy_key,uid=1000 \\ 
    export GIT_SSH_COMMAND="ssh -i /tmp/neurolab_deploy_key -o IdentitiesOnly=yes" && \\
    git pull --rebase
WORKDIR /neurolab/{CONFIG.subproject_name}
COPY --chown=1000:1000 {CONFIG.relative_self_fname} .
'''
    if os.path.exists(CONFIG.initrd_path):
        dockerfile_content += f'''
RUN mkdir -p /neurolab/{CONFIG.relative_initrd_path}
COPY --chown=1000:1000  {CONFIG.relative_initrd_path} /neurolab/{CONFIG.relative_initrd_path}
'''

    dockerfile_content += f'''
RUN touch /neurolab/.docker_launch
CMD ["papermill", "{os.path.basename(CONFIG.self_fname)}", "{os.path.basename(CONFIG.self_fname)}"]
    '''
    with tempfile.NamedTemporaryFile(mode='wt', suffix='.Dockerfile') as dockerfile:
        with open(dockerfile.name, 'w') as f:
            f.write(dockerfile_content)

        image_tag = os.path.join(CONFIG.docker_registry, '${MODEL_NAME}' + ':' + '${MODEL_VERSION}')
        subprocess.run(
            [
                'docker', 'buildx', 'build', '-f', dockerfile.name, '-t', image_tag, build_project_root_path, 
                '--load', '--no-cache', '--network=host', 
                # --add-host is used to overcome problems with VPN (when VPN enabled build process cannot resolve nexus host)
                '--add-host', 'nexus=127.0.0.1',
                # use SSH to connect to GIT to overcome Anti-DDOS barriers
                '--secret', 'id=neurolab_deploy_key,src=/home/misha/.ssh/neurolab_deploy_key',
            ],
            text=True,                # Handles input/output as strings instead of bytes
            capture_output=False,     # Streams Docker's build output directly to your terminal
            check=True                # Raises an exception if the command fails
        )
        subprocess.run(
            ['docker', 'push', image_tag],
            text=True,                # Handles input/output as strings instead of bytes
            capture_output=False,     # Streams Docker's build output directly to your terminal
            check=True                # Raises an exception if the command fails
        )
# @launchit.stop